Sparkov is the **contrast** to the earlier corpus: real `cc_num`, real timestamp, **no pre-engineered feature block** — so the uid features that added no lift there should add lift here.

## 0. Setup

In [ ]:
%load_ext autoreload
%autoreload 2
import sys, json
from pathlib import Path
import pandas as pd
from IPython.display import Image, display
pd.set_option("display.max_columns", 60); pd.set_option("display.width", 180)
MODULE_DIR = Path.cwd()
if str(MODULE_DIR) not in sys.path: sys.path.insert(0, str(MODULE_DIR))
import config, prepare_sparkov, dataset_audit, prepared_loader
from config import SparkovPrepConfig
print("modules loaded from:", MODULE_DIR, "| prep version:", config.PREP_VERSION)

## 1. Configure paths

In [ ]:
cfg = SparkovPrepConfig(data_dir=MODULE_DIR/"data", output_dir=MODULE_DIR/"prepared",
                        figures_dir=MODULE_DIR/"prepared"/"figures")
for f in (cfg.train_file, cfg.test_file):
    p = cfg.data_dir/f
    assert p.exists(), f"MISSING: {p} (put fraudTrain.csv / fraudTest.csv there or fix data_dir)"
print("reading raw from:", cfg.data_dir); print("writing clean to:", cfg.output_dir)
print("raw files found.")

## 2. Build the clean dataset (staged)

### 2.1 Concat train + test

In [ ]:
df = prepare_sparkov.load_and_concat(cfg)
print("concatenated:", df.shape)
df.head(3)

### 2.2 Derive features (hour, dayofweek, age, distance) + uid, then select & sort

In [ ]:
df = prepare_sparkov.add_derived(df, cfg)
df = prepare_sparkov.select_columns(df, cfg)
assert df[cfg.time_col].is_monotonic_increasing
print("after derive + select:", df.shape)
df[[cfg.time_col, cfg.week_col, "hour", "dayofweek", "age", "distance", cfg.uid_col, cfg.amount_col]].head(5)

### 2.3 Downcast + write clean CSV + manifest

In [ ]:
df = prepare_sparkov.reduce_mem_usage(df, cfg)
manifest = prepare_sparkov.write_outputs(df, cfg)
print("wrote:", cfg.clean_csv_path); print("wrote:", cfg.manifest_path)
print(f"clean dataset: {manifest['n_rows']:,} rows x {manifest['n_columns']} cols")

## 3. Audit

In [ ]:
report = dataset_audit.audit(df, cfg)

### 3.1 Headline + flags

In [ ]:
print(f"transactions: {report['n_transactions']:,}")
print(f"fraud rate  : {report['fraud_rate']:.4%}  ({report['n_fraud']:,})")
print(f"weeks       : {report['week_min']}-{report['week_max']} ({report['n_weeks']})")
print("\nuid:"); [print(f"  {k:22s}: {v}") for k,v in report["uid"].items()]
print("\nflags:"); [print("  -", f) for f in report["flags"]]

### 3.2 Tables + figure

In [ ]:
display(pd.read_csv(cfg.output_dir/"cardinality.csv"))
display(pd.read_csv(cfg.output_dir/"weekly_distribution.csv").head(20))
fig = cfg.figures_dir/"weekly_volume_fraud.png"
Image(filename=str(fig)) if fig.exists() else print("no figure")

## 4. Sanity checks

In [ ]:
def check(name, ok, detail=""):
    print(f"[{'PASS ' if ok else 'CHECK'}] {name}" + (f"  ->  {detail}" if detail else ""))
check("fraud rate plausible (0.1%-1.5%)", 0.001 <= report["fraud_rate"] <= 0.02, f"{report['fraud_rate']:.3%}")
check("uid recurs (median txns/uid > 1)", report["uid"]["median_txns_per_uid"] > 1,
      f"median={report['uid']['median_txns_per_uid']}")
check("multiple weeks for folds", report["n_weeks"] >= 8, f"{report['n_weeks']} weeks")

## 5. Handoff to the experiment engine

The `fraud_experiment/` engine is dataset-agnostic. See `README.md` for the
`ExperimentConfig` override that runs the uid-on/off contrast on this output.